# Stripe Subscriptions Upsell — Empirical Holdout Conversion Analysis

**Goal:** Identify non-Subscription merchants with the highest likelihood of converting to Stripe Subscriptions, and rank them for a targeted sales/marketing campaign.

**Approach:** Empirical holdout analysis — we use observed behavior in an 11-month window to characterize merchants, then measure who actually adopted Subscriptions in the following 3-month window. Segments with higher-than-baseline adoption rates become the scoring dimensions.

| Window | Period | Purpose |
|---|---|---|
| Observation | Apr 2041 – Mar 2042 (12 months) | Characterize merchant behavior |
| Conversion | Apr 2042 – Jun 2042 (3 months) | Measure who adopted Subscriptions |

All heavy analytics are performed in **DuckDB SQL**. Python is used only to execute queries and write CSV outputs.

## 0. Setup

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Register CSVs as views
con.execute("""
    CREATE OR REPLACE VIEW merchants AS
        SELECT * FROM read_csv_auto('merchants.csv');

    CREATE OR REPLACE VIEW payments AS
        SELECT * FROM read_csv_auto('payments.csv.csv');
""")

print("Data loaded.")


## 1. Build the Analytical Base

We construct a clean merchant-level table from the raw daily payments data. The pipeline has four steps:

1. **`obs_daily`** — aggregate raw payment rows to one row per merchant per day within the observation window
2. **`obs_monthly`** — roll up daily data to monthly totals (needed for the revenue CV calculation)
3. **`obs_non_sub`** — one row per non-Subscription merchant, with all behavioral features computed
4. **`master`** — join in merchant metadata and the conversion flag

### Key design decisions

**Observation window:** May 1, 2041 – Nov 30, 2041 (7 months). **Conversion window:** Dec 1, 2041 – Jun 22, 2042 (7 months). The 7/7 split was chosen to maximize converters in the holdout window — a longer conversion window yields more observed adoptions and tighter confidence intervals on segment rates.

**`avg_monthly_usd`** — average monthly volume excluding zero-volume months. Used for tier assignment and volume quartile segmentation, as it is a cleaner measure of typical business activity than total volume.

In [ ]:
con.execute("""
-- Step 1: daily aggregates per merchant in observation window
CREATE OR REPLACE VIEW obs_daily AS
    SELECT merchant,
           date::DATE                          AS txn_date,
           SUM(subscription_volume)            AS sub_vol,
           SUM(checkout_volume)                AS checkout_vol,
           SUM(payment_link_volume)            AS paylink_vol,
           SUM(total_volume)                   AS total_vol
    FROM payments
    WHERE date::DATE BETWEEN '2041-04-01' AND '2042-03-31'
    GROUP BY merchant, txn_date;

-- Step 2: monthly rollup per merchant (clean base for CV)
CREATE OR REPLACE VIEW obs_monthly AS
    SELECT merchant,
           DATE_TRUNC('month', txn_date)       AS month,
           SUM(total_vol)                      AS monthly_vol
    FROM obs_daily
    GROUP BY merchant, month;

-- Step 3: per-merchant aggregates
CREATE OR REPLACE VIEW obs_non_sub AS
    WITH daily_agg AS (
        SELECT merchant,
               SUM(sub_vol)                            AS obs_sub_vol,
               SUM(checkout_vol)                       AS obs_checkout_vol,
               SUM(paylink_vol)                        AS obs_paylink_vol,
               SUM(total_vol)                          AS obs_total_vol,
               COUNT(DISTINCT txn_date)                AS active_days
        FROM obs_daily
        GROUP BY merchant
    ),
    monthly_agg AS (
        SELECT merchant,
               COUNT(*)                                AS active_months,
               STDDEV(monthly_vol)                     AS monthly_stddev,
               AVG(monthly_vol)                        AS monthly_avg
        FROM obs_monthly
        GROUP BY merchant
    ),
    nonzero_vol_merchants AS (
        SELECT DISTINCT merchant
        FROM payments
        WHERE total_volume > 0
    )
    SELECT d.merchant,
           d.obs_sub_vol,
           d.obs_checkout_vol,
           d.obs_paylink_vol,
           d.obs_total_vol,
           d.active_days,
           m.active_months,
           m.monthly_stddev,
           m.monthly_avg
    FROM daily_agg d
    JOIN monthly_agg m USING (merchant)
    JOIN nonzero_vol_merchants n USING (merchant)
    WHERE d.obs_sub_vol = 0;

-- Converters: merchants who adopted Subscriptions in the conversion window
CREATE OR REPLACE VIEW converters AS
    SELECT DISTINCT merchant
    FROM payments
    WHERE date::DATE BETWEEN '2042-04-01' AND '2042-06-22'
      AND subscription_volume > 0;

-- Master table: obs attributes + merchant metadata + conversion flag
CREATE OR REPLACE VIEW master AS
    SELECT
        o.merchant,
        m.industry,
        m.business_size,
        m.country,
        m.first_charge_date,
        o.obs_checkout_vol  / 100.0  AS checkout_usd,
        o.obs_paylink_vol   / 100.0  AS paylink_usd,
        (o.obs_checkout_vol + o.obs_paylink_vol) / 100.0 AS combined_vol_usd,
        GREATEST(o.obs_total_vol - o.obs_checkout_vol - o.obs_paylink_vol, 0) / 100.0 AS unattributed_vol_usd,
        CASE
            WHEN GREATEST(o.obs_checkout_vol, o.obs_paylink_vol, GREATEST(o.obs_total_vol - o.obs_checkout_vol - o.obs_paylink_vol, 0)) = o.obs_checkout_vol
                AND o.obs_checkout_vol > 0                          THEN 'checkout'
            WHEN GREATEST(o.obs_checkout_vol, o.obs_paylink_vol, GREATEST(o.obs_total_vol - o.obs_checkout_vol - o.obs_paylink_vol, 0)) = o.obs_paylink_vol
                AND o.obs_paylink_vol > 0                           THEN 'paylink'
            WHEN GREATEST(o.obs_total_vol - o.obs_checkout_vol - o.obs_paylink_vol, 0) > 0 THEN 'unattributed'
            ELSE 'no_volume'
        END AS dominant_method,
        CASE WHEN (o.obs_checkout_vol + o.obs_paylink_vol) > 0
             THEN ROUND(o.obs_paylink_vol::DOUBLE / (o.obs_checkout_vol + o.obs_paylink_vol), 3)
             ELSE NULL END AS paylink_mix,
        o.obs_total_vol     / 100.0  AS total_usd,
        ROUND(o.active_days / 366.0, 3)                     AS active_day_ratio,
        CASE WHEN o.monthly_avg > 0
             THEN ROUND(o.monthly_stddev / o.monthly_avg, 3)
             ELSE NULL END                                   AS revenue_cv,
        CASE
            WHEN m.first_charge_date = '0' OR m.first_charge_date IS NULL THEN NULL
            ELSE ROUND(DATEDIFF('day',
                    TRY_CAST(m.first_charge_date AS TIMESTAMPTZ),
                    TIMESTAMPTZ '2041-04-01 00:00:00+00') / 30.44, 1)
        END                                                  AS tenure_months,
        CASE WHEN c.merchant IS NOT NULL THEN 1 ELSE 0 END  AS converted
    FROM obs_non_sub o
    JOIN merchants m USING (merchant)
    LEFT JOIN converters c USING (merchant);
""")

print("Views created.")

In [ ]:
# Quick sanity check on the master table
con.execute("""
    SELECT
        COUNT(*)                                    AS total_merchants,
        SUM(converted)                              AS total_converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS baseline_conversion_pct,
        ROUND(AVG(total_usd), 0)                    AS avg_total_vol_usd,
        ROUND(MEDIAN(total_usd), 0)                 AS median_total_vol_usd
    FROM master
""").fetchdf()

## 2. Segment Conversion Analysis

We segment non-Subscription merchants along eight dimensions and compute the conversion rate for each segment. This tells us which attributes are predictive of Subscription adoption.

**Confidence intervals** are 95% normal-approximation intervals: `p ± 1.96 × √(p(1−p)/n)`. Segments with n < 30 are excluded as statistically unreliable.

**How to read the results:** A conversion rate well above the baseline (shown below) with a narrow CI is a strong signal. A rate near baseline, or a wide CI, means the attribute is not a reliable predictor.

### Baseline

In [ ]:
baseline_df = con.execute("""
    SELECT COUNT(*) AS n, SUM(converted) AS converters,
           ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS baseline_conversion_pct
    FROM master
""").fetchdf()
baseline_df

### 2a. Industry

Industry is often the strongest predictor of Subscription adoption. Businesses in recurring-friendly verticals — SaaS, education, professional services, healthcare — are structurally more likely to have predictable billing needs that Subscriptions addresses well.

Industries with few merchants (n < 30) are excluded to avoid over-indexing on noisy rates.

In [ ]:
con.execute("""
    SELECT
        industry            AS segment,
        COUNT(*)            AS n,
        SUM(converted)      AS converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct
    FROM master
    GROUP BY industry
    HAVING n >= 30
    ORDER BY conversion_rate_pct DESC
""").fetchdf()

### 2b. Business Size

Business size (small / medium / large) is a coarse proxy for operational complexity. Larger businesses tend to have more sophisticated billing needs, but also longer procurement cycles. Medium businesses often represent the sweet spot: recurring revenue matters but the decision-maker is reachable.

Note: if rates are flat across sizes, size is not a useful targeting dimension on its own and should not be included in the scoring model.

In [ ]:
con.execute("""
    SELECT
        business_size       AS segment,
        COUNT(*)            AS n,
        SUM(converted)      AS converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct
    FROM master
    GROUP BY business_size
    HAVING n >= 30
    ORDER BY conversion_rate_pct DESC
""").fetchdf()

### 2c. Country

Geography can be a meaningful predictor — conversion rates may vary by market maturity, local competitor landscape, or how well Stripe's sales team covers a region. Countries with large merchant populations are the most reliable signals.

Countries with fewer than 30 non-Subscription merchants in the observation window are excluded.

### 2d. Average Monthly Volume Quartile

Average monthly volume (excluding zero-volume months) bucketed into quartiles (Q1 = lowest, Q4 = highest). This is a cleaner measure of typical business activity than total volume, which conflates volume and tenure.

If rates are flat across quartiles, volume is still useful for *prioritization* (bigger merchants = bigger revenue opportunity) but should not be used as a targeting filter on its own.

In [ ]:
con.execute("""
    SELECT
        'Q' || CAST(q AS VARCHAR)                   AS segment,
        COUNT(*)                                    AS n,
        SUM(converted)                              AS converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*)
            - 1.96 * SQRT(SUM(converted) / COUNT(*)
            * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*)
            + 1.96 * SQRT(SUM(converted) / COUNT(*)
            * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct
    FROM (
        SELECT converted, NTILE(4) OVER (ORDER BY avg_monthly_usd) AS q FROM master
    )
    GROUP BY q
    HAVING COUNT(*) >= 30
    ORDER BY segment
""").fetchdf()


### 2e. Dominant Payment Method

This is the most behaviorally direct signal. A merchant's dominant payment method reflects how they currently collect payments and, by extension, how readily they could adopt Subscriptions:

- **checkout** — using Stripe's hosted checkout page; already integrated with Stripe, switching to Subscriptions is a small step
- **paylink** — using Payment Links (no-code); potentially less technically sophisticated, but already on a Stripe-native flow
- **unattributed** — volume not captured by Checkout or Payment Links (likely direct API or other integration); may require more implementation work to adopt Subscriptions
- **no_volume** — no payment volume in the observation window; excluded from conversion scoring

Checkout and paylink merchants are the most natural Subscriptions targets, as they are already using Stripe's product surface.

In [ ]:
con.execute("""
    SELECT
        dominant_method     AS segment,
        COUNT(*)            AS n,
        SUM(converted)      AS converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct
    FROM master
    GROUP BY dominant_method
    HAVING n >= 30
    ORDER BY conversion_rate_pct DESC
""").fetchdf()

### 2f. Payment Link Mix

Among merchants who use both Checkout and Payment Links, what share of their combined volume comes from Payment Links? This captures whether a merchant leans more toward the no-code (Payment Links) or hosted-UI (Checkout) side of Stripe's product.

- `no_combined_vol` — merchant uses neither Checkout nor Payment Links (direct API only)
- `checkout_dominant (>70% checkout)` — primarily using Stripe Checkout
- `mixed (30-70% each)` — meaningful usage of both products
- `paylink_dominant (>70% paylink)` — primarily using Payment Links

This dimension is included in the analysis for completeness, but it is largely redundant with `dominant_method` and was not included in the final scoring model.

In [ ]:
con.execute("""
    SELECT
        CASE
            WHEN paylink_mix IS NULL  THEN 'no_combined_vol'
            WHEN paylink_mix < 0.30   THEN '1_checkout_dominant (>70% checkout)'
            WHEN paylink_mix < 0.70   THEN '2_mixed (30-70% each)'
            ELSE                           '3_paylink_dominant (>70% paylink)'
        END                 AS segment,
        COUNT(*)            AS n,
        SUM(converted)      AS converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct
    FROM master
    GROUP BY segment
    HAVING n >= 30
    ORDER BY segment
""").fetchdf()

### 2g. Active Day Ratio

What fraction of days in the 12-month observation window did a merchant process at least one payment? A high active day ratio indicates consistent, ongoing business activity — a proxy for operational maturity and recurring revenue patterns.

Buckets: low (<25% of days active), medium (25–60%), high (60%+).

**Note:** This dimension showed a weak or inverse relationship with conversion in practice. High-frequency merchants may already have stable payment flows and be less motivated to change. If rates are flat or reversed, this attribute should not be used in the scoring model.

In [ ]:
con.execute("""
    SELECT
        CASE
            WHEN active_day_ratio < 0.25 THEN '1_low (<25%)'
            WHEN active_day_ratio < 0.60 THEN '2_medium (25-60%)'
            ELSE                               '3_high (60%+)'
        END                 AS segment,
        COUNT(*)            AS n,
        SUM(converted)      AS converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct
    FROM master
    GROUP BY segment
    HAVING n >= 30
    ORDER BY segment
""").fetchdf()

### 2h. Revenue Consistency (Coefficient of Variation)

CV = `stddev(monthly_volume) / mean(monthly_volume)` across the observation window. A low CV means stable, predictable monthly revenue — which is a natural fit for Subscriptions. A high CV means highly variable revenue, which may indicate project-based or seasonal work that is harder to put on a subscription model.

Buckets: low variance (CV < 0.5), medium (0.5–1.0), high (1.0+).

**Note:** In practice this dimension showed very flat conversion rates across buckets (roughly 0.7–1.2%), suggesting revenue stability is not a strong standalone predictor in this dataset.

In [ ]:
con.execute("""
    SELECT
        CASE
            WHEN revenue_cv IS NULL        THEN 'no_activity'
            WHEN revenue_cv < 0.5          THEN '1_low_variance (<0.5)'
            WHEN revenue_cv < 1.0          THEN '2_medium_variance (0.5-1.0)'
            ELSE                                '3_high_variance (1.0+)'
        END                 AS segment,
        COUNT(*)            AS n,
        SUM(converted)      AS converters,
        ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct,
        ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct
    FROM master
    GROUP BY segment
    HAVING n >= 30
    ORDER BY segment
""").fetchdf()

### 2i. Merchant Tenure

How long has the merchant been on Stripe as of the start of the observation window? Longer-tenured merchants have had more exposure to Stripe's product suite and may be more receptive to expanding their usage.

Buckets: new (<6 months), mid (6–12 months), mature (12+ months). Merchants with a missing or invalid `first_charge_date` are grouped as `unknown`.

**Data quality note:** 26 merchants have `first_charge_date = '0'` (a sentinel for unknown/missing). These are treated as NULL and assigned to the `unknown` bucket.

## 3. Targeting — CI-Based Segment Filter

### Approach

Rather than building a composite score, we use a statistically conservative filter: a merchant is targeted if they fall into at least one segment where the **95% CI lower bound** of the conversion rate is at or above the baseline rate.

This means we only act on segments where we are confident — even accounting for sampling uncertainty — that the true conversion rate exceeds the overall average. Merchants in segments that don't clear this bar are excluded from the targeting list.

**Qualifying segments (ci_lower ≥ baseline of 1.85%):**

| Dimension | Qualifying segments |
|---|---|
| Industry | Personal services (ci_lower 3.66%), Business services (ci_lower 2.94%) |
| Country | US (2.30%), JP (3.32%), AE (2.73%) |
| Dominant method | Paylink (4.05%) |

A merchant qualifies if they meet **any one** of the above conditions.

### Tiers

Within the qualifying list, we cross conversion signal with revenue opportunity using **average monthly volume** (excluding zero-volume months):

- **Tier 1: High-touch** — top 10% by avg monthly volume among qualifying merchants → dedicated sales outreach
- **Tier 2: Digital campaign** — remaining qualifying merchants → scalable email/in-product campaign
- **Tier 3: Non-qualifying** — below CI threshold on all dimensions → self-serve only

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW targets AS
WITH
vol_cutoff AS (
    -- 90th percentile of avg monthly volume among CI-qualifying merchants only
    SELECT PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY avg_monthly_usd) AS p90
    FROM master
    WHERE industry IN ('Personal services', 'Business services')
       OR country   IN ('US', 'JP', 'AE')
       OR dominant_method = 'paylink'
)
SELECT
    m.merchant,
    m.industry,
    m.business_size,
    m.country,
    m.dominant_method,
    ROUND(m.checkout_usd, 2)       AS checkout_usd,
    ROUND(m.paylink_usd, 2)        AS paylink_usd,
    ROUND(m.total_usd, 2)          AS total_usd,
    ROUND(m.avg_monthly_usd, 2)    AS avg_monthly_usd,
    m.tenure_months,
    m.converted,
    -- qualifying flags
    CASE WHEN m.industry IN ('Personal services', 'Business services') THEN 1 ELSE 0 END AS flag_industry,
    CASE WHEN m.country  IN ('US', 'JP', 'AE')                        THEN 1 ELSE 0 END AS flag_country,
    CASE WHEN m.dominant_method = 'paylink'                           THEN 1 ELSE 0 END AS flag_method,
    CASE
        WHEN (m.industry IN ('Personal services', 'Business services')
           OR m.country   IN ('US', 'JP', 'AE')
           OR m.dominant_method = 'paylink')
         AND m.avg_monthly_usd >= (SELECT p90 FROM vol_cutoff)
            THEN 'Tier 1: High-touch'
        WHEN  m.industry IN ('Personal services', 'Business services')
           OR m.country   IN ('US', 'JP', 'AE')
           OR m.dominant_method = 'paylink'
            THEN 'Tier 2: Digital campaign'
        ELSE    'Tier 3: Non-qualifying'
    END AS tier
FROM master m
ORDER BY
    CASE WHEN m.industry IN ('Personal services','Business services')
              OR m.country IN ('US','JP','AE')
              OR m.dominant_method = 'paylink' THEN 0 ELSE 1 END,
    m.avg_monthly_usd DESC
""")

print("targets view created.")


In [ ]:
# Top 20 qualifying merchants by avg monthly volume
con.execute("""
    SELECT merchant, industry, country, dominant_method,
           avg_monthly_usd, total_usd, tier,
           flag_industry, flag_country, flag_method
    FROM targets
    WHERE tier != 'Tier 3: Non-qualifying'
    LIMIT 20
""").fetchdf()


## 4. Tier Summary

Counts, volume share, and estimated revenue opportunity per tier.

**Incremental revenue estimate:** assumes all of each merchant's current average monthly volume migrates to Subscriptions. Annualized as `avg_monthly_usd × 12`. This is a ceiling estimate — actual incremental subscription revenue will be a fraction of this depending on what share of their payments shift to recurring billing.

In [ ]:
con.execute("""
WITH totals AS (
    SELECT SUM(avg_monthly_usd) AS total_avg_monthly_vol FROM targets
)
SELECT
    tier,
    COUNT(*)                                                        AS merchants,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)             AS pct_merchants,
    ROUND(SUM(avg_monthly_usd), 0)                                  AS total_avg_monthly_vol_usd,
    ROUND(100.0 * SUM(avg_monthly_usd) / MAX(t.total_avg_monthly_vol), 1) AS pct_of_total_vol,
    ROUND(SUM(avg_monthly_usd) * 12, 0)                             AS est_annual_subscription_rev_usd
FROM targets
CROSS JOIN totals t
GROUP BY tier, t.total_avg_monthly_vol
ORDER BY tier
""").fetchdf()


## 5. Avg Monthly Volume Distribution — Qualifying Merchants

Percentile breakdown of avg monthly volume within the qualifying (Tier 1 + Tier 2) merchant list, to understand the volume spread and the Tier 1 cutoff.

In [ ]:
con.execute("""
SELECT
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY avg_monthly_usd), 0) AS p25,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY avg_monthly_usd), 0) AS p50,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY avg_monthly_usd), 0) AS p75,
    ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY avg_monthly_usd), 0) AS p90_tier1_cutoff,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY avg_monthly_usd), 0) AS p95,
    ROUND(MAX(avg_monthly_usd), 0)                                           AS max
FROM targets
WHERE tier != 'Tier 3: Non-qualifying'
""").fetchdf()


In [ ]:
# Export segment_conversion_rates.csv
segment_queries = {
    "industry": "SELECT 'industry' AS attribute, industry AS segment, COUNT(*) AS n, SUM(converted) AS converters, ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct FROM master GROUP BY industry HAVING n >= 30 ORDER BY conversion_rate_pct DESC",
    "business_size": "SELECT 'business_size' AS attribute, business_size AS segment, COUNT(*) AS n, SUM(converted) AS converters, ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct FROM master GROUP BY business_size HAVING n >= 30 ORDER BY conversion_rate_pct DESC",
    "country": "SELECT 'country' AS attribute, country AS segment, COUNT(*) AS n, SUM(converted) AS converters, ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct FROM master GROUP BY country HAVING n >= 30 ORDER BY conversion_rate_pct DESC",
    "dominant_method": "SELECT 'dominant_method' AS attribute, dominant_method AS segment, COUNT(*) AS n, SUM(converted) AS converters, ROUND(100.0 * SUM(converted) / COUNT(*), 2) AS conversion_rate_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) - 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_lower_pct, ROUND(100.0 * (SUM(converted) / COUNT(*) + 1.96 * SQRT(SUM(converted) / COUNT(*) * (1 - SUM(converted) / COUNT(*)) / COUNT(*))), 2) AS ci_upper_pct FROM master GROUP BY dominant_method HAVING n >= 30 ORDER BY conversion_rate_pct DESC",
}

seg_df = pd.concat([con.execute(q).fetchdf() for q in segment_queries.values()], ignore_index=True)
seg_df.to_csv("segment_conversion_rates.csv", index=False)
print("Exported segment_conversion_rates.csv")

# Export targets.csv with tier
con.execute("""
    COPY (
        WITH vol_cutoff AS (
            SELECT PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY total_usd) AS p90
            FROM targets_scored
        )
        SELECT
            ROW_NUMBER() OVER (ORDER BY score DESC) AS rank,
            CASE
                WHEN total_usd >= (SELECT p90 FROM vol_cutoff) AND score >= 150
                    THEN 'Tier 1: High-touch'
                WHEN score >= 150
                    THEN 'Tier 2a: High-intent'
                WHEN total_usd >= (SELECT p90 FROM vol_cutoff)
                    THEN 'Tier 2b: High-value'
                ELSE
                    'Tier 3: Self-serve'
            END AS tier,
            *
        FROM targets_scored
        ORDER BY score DESC
    )
    TO 'targets.csv' (HEADER, DELIMITER ',')
""")
print("Exported targets.csv")